In [ ]:
import copy
import math
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import PowerTransformer, StandardScaler
from torch import nn
from torch.func import functional_call, stack_module_state, vmap

TARGET = "DIC"
SEED = 232
EPOCHS = 1000
FIRST_HIDDEN = 40
SECOND_HIDDEN = 20
N_SEEDS = 20
HUBER_DELTA_UMOL_PER_KG = 5.0
HEAVY_TAILED_COLUMNS = ["R_Depth", "NO2uM", "NH3uM", "R_Nuts"]

SKEWED_REMAINDER_COLUMNS = ["SiO3uM"]
POSITION_COLUMNS = ["Lat_Dec", "Lon_Dec"]
POSITION_WIDTH = 32
VISIT_WIDTH = 256
VISIT_OMEGA0 = 3000.0
VISIT_FEATURE_SCALE = (2 / VISIT_WIDTH) ** 0.5
VISIT_ROWS_AT_HALF_SHRINK = 1.25
DEPTH_COLUMN = "R_Depth"

SURFACE_DEPTH_M = 60.0
SURFACE_LOGISTIC_SCALE_M = 10.0


@dataclass(frozen=True)
class InputLayout:
    position_slots: list[int]
    depth_slot: int
    surface_cut: float
    surface_scale: float


class SkipMLP(nn.Module):
    def __init__(self, n_features: int, layout: InputLayout) -> None:
        super().__init__()
        position_slots = layout.position_slots
        self.hidden = nn.Sequential(
            nn.Linear(n_features, FIRST_HIDDEN),
            nn.GELU(),
            nn.Linear(FIRST_HIDDEN, SECOND_HIDDEN),
            nn.GELU(),
            nn.Linear(SECOND_HIDDEN, 1),
        )
        self.skip = nn.Linear(n_features, 1)
        self.position_slots = position_slots
        self.depth_slot = layout.depth_slot
        # 使っていない層。乱数を引く順番を同じにするため残す
        self.position_waves = nn.Linear(len(position_slots), POSITION_WIDTH)
        self.position_out = nn.Linear(POSITION_WIDTH, 1, bias=False)
        nn.init.zeros_(self.position_out.weight)

        frequencies = torch.randn(len(position_slots), VISIT_WIDTH)
        phases = 2 * torch.pi * torch.rand(VISIT_WIDTH)
        self.visit_waves = nn.Linear(len(position_slots), VISIT_WIDTH)
        with torch.no_grad():
            self.visit_waves.weight.copy_(VISIT_OMEGA0 * frequencies.T)
            self.visit_waves.bias.copy_(phases)
        self.visit_out = nn.Linear(VISIT_WIDTH, 1, bias=False)
        nn.init.zeros_(self.visit_out.weight)
        self.visit_surface_out = nn.Linear(VISIT_WIDTH, 1, bias=False)
        nn.init.zeros_(self.visit_surface_out.weight)
        self.surface_cut = nn.Parameter(torch.tensor([layout.surface_cut]))
        self.surface_log_scale = nn.Parameter(torch.tensor([math.log(layout.surface_scale)]))

    def visit_term(self, x: torch.Tensor) -> torch.Tensor:
        waves = VISIT_FEATURE_SCALE * torch.sin(self.visit_waves(x[:, self.position_slots]))
        surface_scale = torch.exp(self.surface_log_scale)
        surface = torch.sigmoid((self.surface_cut - x[:, [self.depth_slot]]) / surface_scale)
        return self.visit_out(waves) + surface * self.visit_surface_out(waves)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.hidden(x) + self.skip(x) + self.visit_term(x)


def to_tensor(values: np.ndarray) -> torch.Tensor:
    return torch.tensor(values, dtype=torch.float32)


def build_input_transformer() -> ColumnTransformer:
    return ColumnTransformer(
        [("heavy_tails", PowerTransformer(method="yeo-johnson"), HEAVY_TAILED_COLUMNS)],
        remainder=StandardScaler(),
    )


def remainder_slots(transformer: ColumnTransformer, columns: list[str]) -> list[int]:
    output_names = list(transformer.get_feature_names_out())
    return [output_names.index(f"remainder__{column}") for column in columns]


def depths_in_slot_units(
    transformer: ColumnTransformer, train_features: pd.DataFrame, depths_m: list[float]
) -> list[float]:
    probe = train_features[HEAVY_TAILED_COLUMNS].iloc[[0] * len(depths_m)].assign(
        **{DEPTH_COLUMN: depths_m}
    )
    transformed = transformer.named_transformers_["heavy_tails"].transform(probe)
    return transformed[:, HEAVY_TAILED_COLUMNS.index(DEPTH_COLUMN)].tolist()


def input_layout(transformer: ColumnTransformer, train_features: pd.DataFrame) -> InputLayout:
    output_names = list(transformer.get_feature_names_out())
    shallower, cut, deeper = depths_in_slot_units(
        transformer,
        train_features,
        [
            SURFACE_DEPTH_M - SURFACE_LOGISTIC_SCALE_M,
            SURFACE_DEPTH_M,
            SURFACE_DEPTH_M + SURFACE_LOGISTIC_SCALE_M,
        ],
    )
    return InputLayout(
        position_slots=remainder_slots(transformer, POSITION_COLUMNS),
        depth_slot=output_names.index(f"heavy_tails__{DEPTH_COLUMN}"),
        surface_cut=cut,
        surface_scale=(deeper - shallower) / 2,
    )


def fit_transform_inputs(
    train_features: pd.DataFrame, test_features: pd.DataFrame
) -> tuple[np.ndarray, np.ndarray, InputLayout]:
    transformer = build_input_transformer()
    x = transformer.fit_transform(train_features)
    x_test = transformer.transform(test_features)
    slots = remainder_slots(transformer, SKEWED_REMAINDER_COLUMNS)
    skew_transformer = PowerTransformer(method="yeo-johnson")
    x[:, slots] = skew_transformer.fit_transform(train_features[SKEWED_REMAINDER_COLUMNS])
    x_test[:, slots] = skew_transformer.transform(test_features[SKEWED_REMAINDER_COLUMNS])
    return x, x_test, input_layout(transformer, train_features)


def fit_predict(train: pd.DataFrame, test: pd.DataFrame) -> np.ndarray:
    features = list(test.columns)
    x_values, x_test_values, layout = fit_transform_inputs(train[features], test[features])
    x, x_test = to_tensor(x_values), to_tensor(x_test_values)
    y_scaler = StandardScaler()
    y = to_tensor(y_scaler.fit_transform(train[[TARGET]]))

    delta = HUBER_DELTA_UMOL_PER_KG / y_scaler.scale_[0]
    visit_penalty = VISIT_ROWS_AT_HALF_SHRINK / (2 * len(train))
    pred = train_ensemble_and_predict(x, y, x_test, delta, visit_penalty, layout).mean(axis=0)
    return y_scaler.inverse_transform(pred).ravel()


def build_seeded_models(n_features: int, layout: InputLayout) -> list[nn.Module]:
    models = []
    for i in range(N_SEEDS):
        torch.manual_seed(SEED + i)
        models.append(SkipMLP(n_features, layout))
    return models


def train_ensemble_and_predict(
    x: torch.Tensor,
    y: torch.Tensor,
    x_test: torch.Tensor,
    delta: float,
    visit_penalty: float,
    layout: InputLayout,
) -> np.ndarray:
    models = build_seeded_models(x.shape[1], layout)
    params, buffers = stack_module_state(models)
    skeleton = copy.deepcopy(models[0]).to("meta")

    def call_one(p: dict, b: dict, inputs: torch.Tensor) -> torch.Tensor:
        return functional_call(skeleton, (p, b), (inputs,))

    ensemble = vmap(call_one, in_dims=(0, 0, None), randomness="different")
    optimizer = torch.optim.AdamW(params.values(), lr=3e-3, weight_decay=1e-4)
    for _ in range(EPOCHS):
        optimizer.zero_grad()
        pred = ensemble(params, buffers, x)
        losses = nn.functional.huber_loss(pred, y.expand_as(pred), delta=delta, reduction="none")
        visit_weights = (params["visit_out.weight"], params["visit_surface_out.weight"])
        visit_penalty_loss = visit_penalty * sum((w**2).sum() for w in visit_weights)
        (losses.mean(dim=(1, 2)).sum() + visit_penalty_loss).backward()
        optimizer.step()

    with torch.no_grad():
        return ensemble(params, buffers, x_test).numpy()


In [ ]:
from pathlib import Path

kaggle_inputs = sorted(Path("/kaggle/input").rglob("train.csv")) if Path("/kaggle/input").exists() else []
data_dir = kaggle_inputs[0].parent if kaggle_inputs else Path("data/raw")

train = pd.read_csv(data_dir / "train.csv").rename(columns={"TA1.x": "TA1"})
test = pd.read_csv(data_dir / "test.csv")
features = [c for c in test.columns if c != "id" and c in train.columns]
print(len(features), "input columns:", features)

pred = fit_predict(train[[*features, TARGET]], test[features])
submission = pd.DataFrame({"id": test["id"], TARGET: pred})
submission.to_csv("submission.csv", index=False)
submission.head()
